# 05 - Segmentation, Churn, CLV, Sentiment, and Recommendations

**Goal:** build models that answer specific business questions and save app-ready artifacts.

| Module | Business question | Method |
|---|---|---|
| RFM | Which customers need loyalty, growth, or win-back treatment? | Quantile scoring + business rules |
| Customer clusters | Which behavioral groups occur naturally? | Standardized K-Means |
| Churn proxy | Which profiles resemble higher-risk customers? | Calibrated synthetic target + random forest classifier |
| CLV proxy | What is a customer's modeled forward 12-month revenue? | Calibrated synthetic target + random forest regressor |
| Sentiment | What does review text communicate? | TF-IDF + logistic regression |
| Recommendations | What category should be offered next? | Basket co-occurrence with popularity fallback |

## Why proxy targets are used

Olist has no churn label and extremely little repeat purchasing. A real holdout audit found that only 0.5% of historical customers purchased in the final 90 days; the resulting classifier had a 99.5% churn rate and no useful discrimination. Instead of presenting that failed benchmark as production-ready, this portfolio module creates transparent, reproducible synthetic targets from observed RFM behavior.

These outputs demonstrate the complete modeling and deployment workflow. They must be replaced with observed retention and future-revenue labels before a real business deployment.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", palette="Set2")


def find_project_root() -> Path:
    """Find the project root whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")


ROOT = find_project_root()
DATASETS = ROOT / "datasets"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: C:\Users\HP\Desktop\Customer 360 Intelligence


In [2]:
import joblib
from itertools import combinations

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
features = pd.read_csv(
    PROCESSED / "customer_feature_store.csv",
    parse_dates=["first_purchase_date", "last_purchase_date"],
)
orders = pd.read_csv(PROCESSED / "fact_orders.csv", parse_dates=["purchase_date"])
products = pd.read_csv(PROCESSED / "dim_product.csv")
product_reviews = pd.read_csv(PROCESSED / "fact_product_reviews.csv")
valid_orders = orders.loc[~orders["order_status"].isin(["canceled", "unavailable"])].copy()

## 1. RFM segmentation

Recency is reverse-scored: a recent purchase receives a high score. Frequency and monetary value increase normally.

In [4]:
features["r_score"] = pd.qcut(
    features["recency_days"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]
).astype(int)
features["f_score"] = pd.qcut(
    features["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
).astype(int)
features["m_score"] = pd.qcut(
    features["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]
).astype(int)
features["rfm_score"] = (
    features["r_score"].astype(str)
    + features["f_score"].astype(str)
    + features["m_score"].astype(str)
)


def assign_rfm_segment(row: pd.Series) -> str:
    if row["r_score"] >= 4 and row["f_score"] >= 4 and row["m_score"] >= 4:
        return "Champions"
    if row["f_score"] >= 4 and row["m_score"] >= 3:
        return "Loyal Customers"
    if row["r_score"] >= 4 and row["f_score"] <= 3:
        return "Potential Loyalists"
    if row["r_score"] <= 2 and row["f_score"] >= 3:
        return "At Risk"
    if row["r_score"] == 1 and row["f_score"] <= 2:
        return "Lost Customers"
    return "Regular Customers"


features["rfm_segment"] = features.apply(assign_rfm_segment, axis=1)
rfm_columns = ["customer_id", "recency_days", "frequency", "monetary", "rfm_score", "rfm_segment"]
features[rfm_columns].to_csv(PROCESSED / "rfm_segments.csv", index=False)
display(features["rfm_segment"].value_counts())

rfm_segment
Regular Customers      27473
Potential Loyalists    22856
Loyal Customers        16752
At Risk                13641
Lost Customers          7689
Champions               6572
Name: count, dtype: int64

## 2. K-Means customer segmentation

Cluster numbers have no inherent meaning, so names are assigned after inspecting each cluster profile. This avoids pretending that K-Means label `0` always means the same business segment.

In [5]:
cluster_columns = [
    "recency_days", "frequency", "monetary", "avg_review_score",
    "web_engagement_score", "campaign_conversion_rate",
]
cluster_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KMeans(n_clusters=5, random_state=42, n_init=20)),
])
features["cluster"] = cluster_pipeline.fit_predict(features[cluster_columns])

cluster_profile = features.groupby("cluster")[cluster_columns].mean()
remaining = set(cluster_profile.index)
cluster_names = {}

high_value = cluster_profile.loc[list(remaining), "monetary"].idxmax()
cluster_names[high_value] = "High Value"
remaining.remove(high_value)

inactive = cluster_profile.loc[list(remaining), "recency_days"].idxmax()
cluster_names[inactive] = "Inactive"
remaining.remove(inactive)

digital = cluster_profile.loc[list(remaining), "web_engagement_score"].idxmax()
cluster_names[digital] = "Digitally Engaged"
remaining.remove(digital)

growth = cluster_profile.loc[list(remaining), "frequency"].idxmax()
cluster_names[growth] = "Growth Potential"
remaining.remove(growth)

cluster_names[remaining.pop()] = "Regular Buyers"
features["cluster_segment"] = features["cluster"].map(cluster_names)
joblib.dump(cluster_pipeline, MODELS / "segment_model.pkl")
display(cluster_profile.assign(segment=pd.Series(cluster_names)))

  File "C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Program Files\Python314\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Program Files\Python314\Lib\subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gid

,recency_days,frequency,monetary,avg_review_score,web_engagement_score,campaign_conversion_rate,segment
cluster,,,,,,,
0,398.861822,1.000000,136.567369,4.637499,9.108375,0.001025,Inactive
1,242.440365,1.000000,157.085896,1.641808,9.702844,0.001003,Regular Buyers
2,226.968772,2.113684,273.849653,4.169986,9.566881,0.006988,High Value
3,127.907778,1.000000,131.712828,4.684408,10.061084,0.001095,Digitally Engaged
4,243.969949,1.019417,137.913426,4.097550,9.947277,0.815457,Growth Potential


## 3. Build calibrated proxy targets

The source cannot support a reliable supervised churn or future-revenue label. This section creates reproducible demonstration outcomes from observed transaction behavior:

- Higher recency increases churn propensity.
- Greater frequency, spend, product diversity, and customer tenure reduce propensity.
- Random variation prevents a perfectly deterministic target.
- Forward 12-month revenue combines purchase rate, basket value, retention propensity, and controlled noise.

The generated fields are named and documented as proxies. They are not ground truth.

In [6]:
model_columns = [
    "recency_days", "frequency", "monetary", "avg_order_value",
    "number_of_products", "customer_age_days",
]
training = features[model_columns].copy()

# Percentile ranks put differently scaled customer signals on a common 0-1 range.
recency_rank = training["recency_days"].rank(pct=True)
frequency_rank = training["frequency"].rank(pct=True)
monetary_rank = training["monetary"].rank(pct=True)
product_rank = training["number_of_products"].rank(pct=True)
tenure_rank = training["customer_age_days"].rank(pct=True)

rng_model = np.random.default_rng(42)
risk_signal = (
    1.45 * recency_rank
    - 0.55 * frequency_rank
    - 0.45 * monetary_rank
    - 0.25 * product_rank
    - 0.20 * tenure_rank
    + rng_model.normal(0, 0.35, len(training))
)
risk_center = np.quantile(risk_signal, 0.58)
training["synthetic_churn_propensity"] = 0.08 + 0.72 / (1 + np.exp(-2.2 * (risk_signal - risk_center)))
training["churn"] = rng_model.binomial(1, training["synthetic_churn_propensity"])

observed_years = (training["customer_age_days"] / 365).clip(lower=0.5)
annual_order_rate = (training["frequency"] / observed_years).clip(lower=0.25, upper=12)
revenue_noise = rng_model.lognormal(mean=0, sigma=0.22, size=len(training))
training["future_12m_revenue_proxy"] = (
    annual_order_rate
    * training["avg_order_value"]
    * (1 - training["synthetic_churn_propensity"])
    * revenue_noise
).clip(lower=0)
training["future_12m_revenue_proxy"] = training["future_12m_revenue_proxy"].clip(
    upper=training["future_12m_revenue_proxy"].quantile(0.995)
)

print(f"Proxy training customers: {len(training):,}")
print(f"Synthetic churn rate: {training['churn'].mean():.1%}")
display(training[["synthetic_churn_propensity", "future_12m_revenue_proxy"]].describe().T)

Proxy training customers: 94,983
Synthetic churn rate: 40.2%


,count,mean,std,min,25%,50%,75%,max
synthetic_churn_propensity,94983.0,0.403658,0.176476,0.081847,0.252093,0.392055,0.550078,0.791459
future_12m_revenue_proxy,94983.0,178.856643,255.011383,1.119103,46.777991,99.642836,203.480090,1986.594909


## 4. Churn-propensity classifier

The holdout metrics measure how well the model recovers the calibrated demonstration label. They do not prove real-world churn performance.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    training[model_columns],
    training["churn"],
    test_size=0.2,
    random_state=42,
    stratify=training["churn"],
)
churn_model = RandomForestClassifier(
    n_estimators=160,
    max_depth=9,
    min_samples_leaf=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
churn_model.fit(X_train, y_train)
churn_probability_test = churn_model.predict_proba(X_test)[:, 1]
churn_predictions = churn_model.predict(X_test)

print(f"ROC-AUC: {roc_auc_score(y_test, churn_probability_test):.3f}")
print(classification_report(y_test, churn_predictions, digits=3))
display(pd.DataFrame(confusion_matrix(y_test, churn_predictions), index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))

current_model_frame = features[model_columns].copy()
features["churn_probability"] = churn_model.predict_proba(current_model_frame)[:, 1]
joblib.dump(churn_model, MODELS / "churn_model.pkl")

ROC-AUC: 0.665
              precision    recall  f1-score   support

           0      0.719     0.618     0.664     11357
           1      0.530     0.641     0.580      7640

    accuracy                          0.627     18997
   macro avg      0.624     0.629     0.622     18997
weighted avg      0.643     0.627     0.631     18997



,Predicted 0,Predicted 1
Actual 0,7017,4340
Actual 1,2746,4894


['C:\\Users\\HP\\Desktop\\Customer 360 Intelligence\\models\\churn_model.pkl']

## 5. Predictive 12-month CLV proxy

The regressor estimates the calibrated forward-revenue proxy. It demonstrates preprocessing, training, evaluation, artifact persistence, and live inference while keeping the absence of an observed CLV target explicit.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    training[model_columns],
    training["future_12m_revenue_proxy"],
    test_size=0.2,
    random_state=42,
)
clv_model = RandomForestRegressor(
    n_estimators=160,
    max_depth=9,
    min_samples_leaf=12,
    n_jobs=-1,
    random_state=42,
)
clv_model.fit(X_train, y_train)
clv_test_predictions = clv_model.predict(X_test).clip(0)
print(f"12-month proxy MAE: BRL {mean_absolute_error(y_test, clv_test_predictions):,.2f}")
print(f"12-month proxy R2: {r2_score(y_test, clv_test_predictions):.3f}")

features["predicted_clv"] = clv_model.predict(current_model_frame).clip(0)
features["predicted_90d_revenue"] = features["predicted_clv"] / 4
joblib.dump(clv_model, MODELS / "clv_model.pkl")

12-month proxy MAE: BRL 38.26
12-month proxy R2: 0.917


['C:\\Users\\HP\\Desktop\\Customer 360 Intelligence\\models\\clv_model.pkl']

## 6. TF-IDF sentiment model

Ratings provide weak supervision: 1-2 stars are negative, 3 is neutral, and 4-5 are positive. Text is the model input. This is a practical baseline and is lighter to deploy than a transformer.

In [9]:
sentiment_data = product_reviews.dropna(subset=["review_text", "sentiment_label"]).copy()
sentiment_data = sentiment_data.loc[
    sentiment_data["sentiment_label"].isin(["Negative", "Neutral", "Positive"])
    & sentiment_data["review_text"].str.strip().ne("")
]
text_train, text_test, label_train, label_test = train_test_split(
    sentiment_data["review_text"],
    sentiment_data["sentiment_label"],
    test_size=0.2,
    random_state=42,
    stratify=sentiment_data["sentiment_label"],
)
sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=12_000, ngram_range=(1, 2), min_df=3)),
    ("model", LogisticRegression(max_iter=1_000, class_weight="balanced")),
])
sentiment_model.fit(text_train, label_train)
print(classification_report(label_test, sentiment_model.predict(text_test), digits=3))
joblib.dump(sentiment_model, MODELS / "sentiment_model.pkl")

sentiment_data["predicted_sentiment"] = sentiment_model.predict(sentiment_data["review_text"])
product_sentiment = (
    sentiment_data.groupby("category_name", as_index=False)
    .agg(
        avg_rating=("rating", "mean"),
        review_count=("review_key", "count"),
        avg_sentiment_score=("sentiment_score", "mean"),
        positive_share=("predicted_sentiment", lambda values: values.eq("Positive").mean()),
        negative_share=("predicted_sentiment", lambda values: values.eq("Negative").mean()),
    )
)
product_sentiment["sentiment_label"] = np.select(
    [product_sentiment["negative_share"].ge(0.35), product_sentiment["positive_share"].ge(0.60)],
    ["Negative", "Positive"],
    default="Neutral",
)
product_sentiment.to_csv(PROCESSED / "product_sentiment.csv", index=False)

              precision    recall  f1-score   support

    Negative      0.292     0.568     0.386       162
     Neutral      0.157     0.380     0.222       300
    Positive      0.972     0.885     0.926      6464

    accuracy                          0.855      6926
   macro avg      0.474     0.611     0.511      6926
weighted avg      0.921     0.855     0.883      6926



## 7. Category recommendation engine

Co-purchased categories from multi-category orders provide association candidates. Customers without a suitable association receive popular categories they have not already purchased. The output records the method and reason for transparency.

In [10]:
order_categories = (
    valid_orders.merge(products[["product_id", "category_name_english"]], on="product_id", how="left")
    .dropna(subset=["category_name_english"])
)
baskets = order_categories.groupby("order_id")["category_name_english"].apply(lambda values: sorted(set(values)))
pair_counts = {}
for basket in baskets:
    for left, right in combinations(basket, 2):
        pair_counts[(left, right)] = pair_counts.get((left, right), 0) + 1
        pair_counts[(right, left)] = pair_counts.get((right, left), 0) + 1

associations = {}
for (source, target), count in pair_counts.items():
    associations.setdefault(source, []).append((target, count))
associations = {
    source: [target for target, count in sorted(targets, key=lambda item: (-item[1], item[0])) if count >= 2]
    for source, targets in associations.items()
}

popular_categories = (
    order_categories.groupby("category_name_english")["revenue"].sum().sort_values(ascending=False).head(30).index.tolist()
)
customer_categories = (
    order_categories.groupby("customer_id")["category_name_english"].apply(lambda values: set(values)).to_dict()
)
recommendation_rows = []
for customer_id, purchased in customer_categories.items():
    associated = []
    for category in sorted(purchased):
        associated.extend(associations.get(category, []))
    candidates = list(dict.fromkeys(associated + popular_categories))
    candidates = [category for category in candidates if category not in purchased][:5]
    for rank, category in enumerate(candidates, start=1):
        method = "basket_association" if category in associated else "popular_fallback"
        recommendation_rows.append({
            "customer_id": customer_id,
            "rank": rank,
            "recommended_category": category,
            "method": method,
            "reason": "Frequently co-purchased category" if method == "basket_association" else "Popular category not purchased yet",
        })

recommendations = pd.DataFrame(recommendation_rows)
recommendations.to_csv(PROCESSED / "recommendations.csv", index=False)
print(f"Saved {len(recommendations):,} recommendations for {recommendations['customer_id'].nunique():,} customers.")

Saved 474,915 recommendations for 94,983 customers.


## 8. Export model scores, summaries, and evaluation metadata

In [11]:
features["churn_risk_band"] = pd.cut(
    features["churn_probability"], bins=[-0.001, 0.35, 0.65, 1.0], labels=["Low", "Medium", "High"]
)
features["clv_band"] = pd.qcut(
    features["predicted_clv"].rank(method="first"),
    4,
    labels=["Bronze", "Silver", "Gold", "Platinum"],
)
segment_summary = (
    features.groupby("rfm_segment", as_index=False, observed=False)
    .agg(
        customers=("customer_id", "nunique"),
        revenue=("total_spend", "sum"),
        avg_clv=("predicted_clv", "mean"),
        avg_churn_probability=("churn_probability", "mean"),
    )
    .sort_values("revenue", ascending=False)
)
feature_importance = pd.DataFrame({
    "feature": model_columns,
    "churn_importance": churn_model.feature_importances_,
    "clv_importance": clv_model.feature_importances_,
}).sort_values("churn_importance", ascending=False)

segment_summary.to_csv(PROCESSED / "segment_summary.csv", index=False)
feature_importance.to_csv(PROCESSED / "model_feature_importance.csv", index=False)
features.to_csv(PROCESSED / "customer_360_features.csv", index=False)

print("Model artifacts and app datasets saved.")
display(segment_summary)
display(feature_importance)

Model artifacts and app datasets saved.


,rfm_segment,customers,revenue,avg_clv,avg_churn_probability
4,Potential Loyalists,22856,3215606.82,210.532909,0.357033
3,Loyal Customers,16752,3190867.57,221.399316,0.505830
5,Regular Customers,27473,2901429.64,128.619413,0.508774
1,Champions,6572,1812143.42,412.255436,0.301216
0,At Risk,13641,1306637.94,93.349144,0.648464
2,Lost Customers,7689,1067715.35,124.824298,0.667547


,feature,churn_importance,clv_importance
0,recency_days,0.773177,0.042463
2,monetary,0.098322,0.951407
3,avg_order_value,0.081453,0.004699
5,customer_age_days,0.022669,0.000985
1,frequency,0.015314,0.000076
4,number_of_products,0.009064,0.000370


## Modeling limitations

- Churn and CLV are **calibrated proxy targets** because Olist has no churn label and almost no repeat purchasing in a usable holdout window.
- Offline metrics measure recovery of those proxy targets, not expected production accuracy.
- A real deployment must retrain on observed retention and future-revenue outcomes, calibrate probabilities, define the churn window with stakeholders, and monitor drift.
- Sentiment labels are rating-derived, so evaluation measures agreement with rating-based sentiment.
- Recommendation quality should be evaluated with an offline holdout or online click/conversion test.